# Data Exploration (Simple Version)
A beginner-friendly walkthrough of the raw CEMS data.  
Same findings as `01_exploration.ipynb`, just simpler code.

---

In [1]:
import pandas as pd
import numpy as np

# load all 5 datasets
raw       = pd.read_csv('../datasets/dev/raw_cems_data.csv')
sensors   = pd.read_csv('../datasets/dev/sensor_master.csv')
maint     = pd.read_csv('../datasets/dev/maintenance_logs.csv')
manual    = pd.read_csv('../datasets/dev/manual_entries.csv')
thresholds = pd.read_csv('../datasets/dev/regulatory_thresholds.csv')

print('Loaded!')
print(f'raw_cems_data  : {raw.shape}')
print(f'sensor_master  : {sensors.shape}')
print(f'maintenance_logs: {maint.shape}')
print(f'manual_entries : {manual.shape}')
print(f'reg_thresholds : {thresholds.shape}')

Loaded!
raw_cems_data  : (5650, 11)
sensor_master  : (10, 9)
maintenance_logs: (6, 5)
manual_entries : (10, 5)
reg_thresholds : (6, 3)


C:\Users\KIIT\AppData\Local\Temp\ipykernel_28292\1440966461.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1. First Look

In [2]:
# what columns do we have and what are their types?
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5650 entries, 0 to 5649
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Record_ID        5650 non-null   object 
 1   Plant_ID         5650 non-null   object 
 2   Stack_ID         5650 non-null   object 
 3   Flow_Rate_m3_hr  4536 non-null   float64
 4   TS               5650 non-null   object 
 5   PM2.5            5564 non-null   object 
 6   SO2              5545 non-null   object 
 7   NOx              5578 non-null   object 
 8   Unit             5650 non-null   object 
 9   Status           5624 non-null   object 
 10  Lat_Lon          5650 non-null   object 
dtypes: float64(1), object(10)
memory usage: 485.7+ KB


In [3]:
# first few rows
raw.head(15)

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
0,E00001,PL-01,S-01,2201.9,01-01-2025 00:00,125.3,NaN,-3.4,ug/m3,OK,"13.0834,80.271"
1,E00002,PL-01,S-02,1881.1,01/01/2025 00:00,153.7,30.5,92.7,ug/m3,maint,"13.0834,80.2721"
2,E00004,PL-03,S-01,1712.5,01-01-2025 00:00,261.4,200.3,7777.0,mg/Nm3,OK,"12.9726,77.5952"
3,E00005,PL-03,S-02,2838.9,01-01-2025 00:00,103.8,118.6,211.1,ug/m3,ok,"12.9718,77.5958"
4,E00007,LOC-DEL-01,A-01,NaN,01-01-2025 00:00,70.5,25.2,139.9,ug/m3,OK,"28.6137,77.2081"
5,E00008,LOC-DEL-02,A-02,0.0,01-01-2025 00:00,121.7,157.7,140.5,ug/m3,FAULT,"28.6205,77.2159"
6,E00009,LOC-MUM-01,A-01,NaN,01-01-2025 00:00,97.3,45.1,BDL,µg/m³,OK,"19.0764,72.8781"
7,E00010,LOC-BLR-01,A-01,0.0,01-01-2025 00:00,195.4,63.5,211.0,ug/m3,MAINT,"12.9726,77.5939"
8,E00011,PL-01,S-01,2077.8,01-01-2025 00:15,149.5,211.0,50.4,µg/m³,FAULT,"13.0819,80.2702"
9,E00012,PL-01,S-02,1321.2,01-01-2025 00:15,183.3,88.2,203.9,ug/m3,FAULT,"13.0829,80.2708"


## 2. Null Values
Which columns have missing data?

In [4]:
# count of null values per column
print(raw.isnull().sum())
print(f'\nTotal rows: {len(raw)}')

Record_ID             0
Plant_ID              0
Stack_ID              0
Flow_Rate_m3_hr    1114
TS                    0
PM2.5                86
SO2                 105
NOx                  72
Unit                  0
Status               26
Lat_Lon               0
dtype: int64

Total rows: 5650


In [5]:
# also check for empty strings (these show up as non-null but are still missing)
for col in raw.columns:
    empty_count = (raw[col].astype(str).str.strip() == '').sum()
    if empty_count > 0:
        print(f'{col}: {empty_count} empty strings')

## 3. Timestamps
The `TS` column should be in a consistent format. Let's see what we actually have.

In [6]:
# look at some unique timestamp patterns
print('Sample timestamps:')
for ts in raw['TS'].sample(20, random_state=42):
    print(f'  "{ts}"')

Sample timestamps:
  "04-01-2025 17:15 UTC"
  "05-01-2025 14:15"
  "31-12-2024 24:00"
  "05/01/2025 18:45"
  "05-01-2025 05:45"
  "01-01-2025 04:15"
  "01-01-2025 14:30"
  "05-01-2025 05:30"
  "05-01-2025 02:00"
  "06-01-2025 15:30"
  "05-01-2025 23:45"
  "03-01-2025 23:15"
  "05-01-2025 01:30"
  "06-01-2025 00:15"
  "01-01-2025 10:45"
  "05-01-2025 04:30"
  "05-01-2025 10:30"
  "06-01-2025 19:15"
  "04/01/2025 16:45"
  "02-01-2025 04:15"


In [7]:
# count: how many have hour=24?
has_24 = raw['TS'].str.contains('24:', na=False)
print(f'Hour=24 timestamps: {has_24.sum()}')

# count: how many have UTC?
has_utc = raw['TS'].str.contains('UTC', na=False)
print(f'UTC timestamps: {has_utc.sum()}')

# count: how many use slash format (DD/MM/YYYY)?
has_slash = raw['TS'].str.contains('/', na=False)
print(f'Slash format: {has_slash.sum()}')

# count: how many are ISO format (starts with year like 2025-)?
has_iso = raw['TS'].str.match(r'^\d{4}-', na=False)
print(f'ISO format: {has_iso.sum()}')

# the rest are standard DD-MM-YYYY
standard = ~has_24 & ~has_utc & ~has_slash & ~has_iso
print(f'Standard format: {standard.sum()}')
print(f'\nTotal: {len(raw)}')

Hour=24 timestamps: 155
UTC timestamps: 283
Slash format: 269
ISO format: 96
Standard format: 4847

Total: 5650


In [8]:
# show examples of each
if has_24.any():
    print('Hour=24 examples:', raw.loc[has_24, 'TS'].head(3).tolist())
if has_utc.any():
    print('UTC examples:', raw.loc[has_utc, 'TS'].head(3).tolist())
if has_slash.any():
    print('Slash examples:', raw.loc[has_slash, 'TS'].head(3).tolist())
if has_iso.any():
    print('ISO examples:', raw.loc[has_iso, 'TS'].head(3).tolist())

Hour=24 examples: ['31-12-2024 24:45', '31-12-2024 24:15', '31-12-2024 24:15']
UTC examples: ['31-12-2024 19:30 UTC', '31-12-2024 20:30 UTC', '31-12-2024 21:30 UTC']
Slash examples: ['01/01/2025 00:00', '01/01/2025 00:30', '01/01/2025 00:30']
ISO examples: ['2025-01-01 02:00:00', '2025-01-01 03:30:00', '2025-01-01 05:15:00']


## 4. Pollutants (PM2.5, SO2, NOx)
These should be numbers, but let's check what's actually in there.

In [9]:
# check the data types — if 'object', there are strings mixed in
for col in ['PM2.5', 'SO2', 'NOx']:
    print(f'{col} → dtype: {raw[col].dtype}')

PM2.5 → dtype: object
SO2 → dtype: object
NOx → dtype: object


In [10]:
# try converting to numbers — anything that fails is a string issue
for col in ['PM2.5', 'SO2', 'NOx']:
    numeric = pd.to_numeric(raw[col], errors='coerce')  # non-numbers become NaN
    
    # how many couldn't be converted?
    failed = numeric.isna() & raw[col].notna() & (raw[col].astype(str).str.strip() != '')
    
    # how many are negative?
    negatives = (numeric < 0).sum()
    
    # how many are suspiciously high (spikes)?
    spikes = (numeric > 5000).sum()
    
    # how many are empty/null?
    empty = (raw[col].astype(str).str.strip() == '').sum() + raw[col].isna().sum()
    
    print(f'\n{col}:')
    print(f'  Non-numeric strings (BDL etc): {failed.sum()}')
    print(f'  Negative values:               {negatives}')
    print(f'  Spikes (>5000):                {spikes}')
    print(f'  Empty/null:                    {empty}')


PM2.5:
  Non-numeric strings (BDL etc): 274
  Negative values:               314
  Spikes (>5000):                158
  Empty/null:                    86

SO2:
  Non-numeric strings (BDL etc): 281
  Negative values:               340
  Spikes (>5000):                159
  Empty/null:                    105

NOx:
  Non-numeric strings (BDL etc): 302
  Negative values:               266
  Spikes (>5000):                168
  Empty/null:                    72


In [11]:
# what do the non-numeric strings look like?
for col in ['PM2.5', 'SO2', 'NOx']:
    numeric = pd.to_numeric(raw[col], errors='coerce')
    non_numeric = raw.loc[numeric.isna() & raw[col].notna() & (raw[col].astype(str).str.strip() != ''), col]
    if len(non_numeric) > 0:
        print(f'{col} non-numeric samples: {non_numeric.head(5).tolist()}')

PM2.5 non-numeric samples: ['<2.0', 'BDL', 'BDL', 'BDL', '< 2.0']
SO2 non-numeric samples: ['< 5.0', '< 2.0', 'BDL', '<5.0', '<5.0']
NOx non-numeric samples: ['BDL', 'BDL', '< 5.0', 'BDL', '<5.0']


## 5. Unit Column

In [12]:
# what unit values are in the data?
print(raw['Unit'].value_counts())

Unit
ug/m3     4521
mg/Nm3     566
µg/m³      563
Name: count, dtype: int64


## 6. Status Column

In [13]:
# what status values are in the data?
print(raw['Status'].value_counts())

Status
MAINT          1320
OK             1311
FAULT          1307
  MAINT         142
Error 404       130
 fault          130
maintenance     127
ok              126
error           123
 Ok             122
  OK            121
 ok             121
 maint          115
offline         110
OFFLINE         108
DOWN            108
down            103
Name: count, dtype: int64


## 7. Lat/Lon

In [14]:
# check for obvious issues: semicolons, impossible values
has_semicolon = raw['Lat_Lon'].str.contains(';', na=False)
has_negative999 = raw['Lat_Lon'].str.contains('-999', na=False)

print(f'Semicolon separator: {has_semicolon.sum()}')
print(f'Impossible coords (-999): {has_negative999.sum()}')

if has_semicolon.any():
    print(f'\nSemicolon examples: {raw.loc[has_semicolon, "Lat_Lon"].head(3).tolist()}')
if has_negative999.any():
    print(f'Impossible examples: {raw.loc[has_negative999, "Lat_Lon"].head(3).tolist()}')

Semicolon separator: 28
Impossible coords (-999): 30

Semicolon examples: ['12.9718;77.5948', '28.62;77.215', '19.076;72.8777']
Impossible examples: ['12.9716,-999.0', '28.6139,-999.0', '28.6139,-999.0']


## 8. Record ID Gaps

In [15]:
# extract the number from Record_ID (E00001 → 1)
record_nums = raw['Record_ID'].str.replace('E', '').astype(int)

# what's the expected range?
expected = set(range(record_nums.min(), record_nums.max() + 1))
actual = set(record_nums)
missing = expected - actual

print(f'Expected IDs: {len(expected)}')
print(f'Actual IDs:   {len(actual)}')
print(f'Missing IDs:  {len(missing)} (these are gaps that need filling)')

if missing:
    sample = sorted(list(missing))[:10]
    print(f'First 10 missing: {[f"E{x:05d}" for x in sample]}')

Expected IDs: 6000
Actual IDs:   5650
Missing IDs:  350 (these are gaps that need filling)
First 10 missing: ['E00003', 'E00006', 'E00093', 'E00137', 'E00162', 'E00163', 'E00173', 'E00189', 'E00203', 'E00225']


## 9. Duplicate Check

In [16]:
# any duplicate rows based on Plant_ID + TS?
dupes = raw.duplicated(subset=['Plant_ID', 'TS'], keep=False)
print(f'Duplicate Plant_ID + TS combos: {dupes.sum()}')

if dupes.any():
    print(raw[dupes].head(10))

Duplicate Plant_ID + TS combos: 2314
   Record_ID Plant_ID Stack_ID  Flow_Rate_m3_hr                TS  PM2.5  \
2     E00004    PL-03     S-01           1712.5  01-01-2025 00:00  261.4   
3     E00005    PL-03     S-02           2838.9  01-01-2025 00:00  103.8   
8     E00011    PL-01     S-01           2077.8  01-01-2025 00:15  149.5   
9     E00012    PL-01     S-02           1321.2  01-01-2025 00:15  183.3   
11    E00014    PL-03     S-01           1878.3  01-01-2025 00:15  151.1   
12    E00015    PL-03     S-02           1598.0  01-01-2025 00:15  278.1   
13    E00016    PL-03     S-03           1261.4  01-01-2025 00:15   -3.2   
22    E00025    PL-03     S-02           2045.6  01-01-2025 00:30   -2.0   
23    E00026    PL-03     S-03           2526.3  01-01-2025 00:30  113.4   
28    E00031    PL-01     S-01           4913.3  01-01-2025 00:45   <2.0   

      SO2     NOx    Unit Status          Lat_Lon  
2   200.3  7777.0  mg/Nm3     OK  12.9726,77.5952  
3   118.6   211.1   ug

## 10. Reference Tables

In [17]:
# sensor_master — the lookup table
print('=== Sensor Master (10 sensors) ===')
print(sensors.to_string(index=False))

=== Sensor Master (10 sensors) ===
  Plant_ID Stack_ID Source_Type Sector  Zero_Drift  Span_Mult  LOD_PM25  LOD_SO2  LOD_NOx
     PL-01     S-01       Stack Cement         0.5       1.02       2.0      4.0      5.0
     PL-01     S-02       Stack Cement         0.2       0.98       2.0      4.0      5.0
     PL-02     S-01       Stack  Steel        -0.5       1.05       2.0      5.0      5.0
     PL-03     S-01       Stack  Power         0.8       1.10       5.0      5.0      5.0
     PL-03     S-02       Stack  Power         0.1       1.00       5.0      5.0      5.0
     PL-03     S-03       Stack  Power         0.0       1.01       5.0      5.0      5.0
LOC-DEL-01     A-01     Ambient   City         0.1       1.00       1.0      2.0      2.0
LOC-DEL-02     A-02     Ambient   City         0.0       0.99       1.0      2.0      2.0
LOC-MUM-01     A-01     Ambient   City        -0.1       1.02       1.0      2.0      2.0
LOC-BLR-01     A-01     Ambient   City         0.2       1.00    

In [18]:
# maintenance_logs — when sensors were being serviced
print('=== Maintenance Logs ===')
print(maint.to_string(index=False))

=== Maintenance Logs ===
  Plant_ID Stack_ID      Maint_Start        Maint_End   Technician
     PL-03     S-02 06-01-2025 05:00 06-01-2025 09:00   Ravi Kumar
     PL-01     S-02 01-01-2025 02:00 01-01-2025 04:00 Suresh Patel
LOC-BLR-01     A-01 01-01-2025 10:00 01-01-2025 14:00 Vikram Singh
LOC-MUM-01     A-01 06-01-2025 08:00 06-01-2025 10:00  Anita Verma
     PL-03     S-01 03-01-2025 20:00 03-01-2025 23:00 Vikram Singh
LOC-MUM-01     A-01 04-01-2025 03:00 04-01-2025 06:00 Suresh Patel


In [19]:
# regulatory_thresholds — legal limits
print('=== Legal Limits ===')
print(thresholds.to_string(index=False))

=== Legal Limits ===
Pollutant Source_Type  Legal_Limit_ugm3
    PM2.5     Ambient              60.0
    PM2.5       Stack             150.0
      SO2     Ambient              80.0
      SO2       Stack             200.0
      NOx     Ambient              80.0
      NOx       Stack             400.0


## 11. Manual Entries — QC & PII

In [20]:
# double-entry check: do Entry1 and Entry2 match?
manual['Diff_%'] = abs(manual['Lab_PM25_Entry1'] - manual['Lab_PM25_Entry2']) / manual['Lab_PM25_Entry1'] * 100

print('Lab PM2.5 double-entry comparison:')
print(manual[['Log_ID', 'Lab_PM25_Entry1', 'Lab_PM25_Entry2', 'Diff_%']].to_string(index=False))

qc_fail = manual['Diff_%'] > 1.0
print(f'\nQC Failures (>1% difference): {qc_fail.sum()} out of {len(manual)}')

Lab PM2.5 double-entry comparison:
Log_ID  Lab_PM25_Entry1  Lab_PM25_Entry2    Diff_%
 L0001             75.8             76.0  0.263852
 L0002             71.7             10.3 85.634589
 L0003            196.1            196.0  0.050994
 L0004            101.0            101.3  0.297030
 L0005             19.6             19.4  1.020408
 L0006            163.9            164.1  0.122026
 L0007             69.4             69.1  0.432277
 L0008            128.1            128.1  0.000000
 L0009             62.4             62.4  0.000000
 L0010             58.0             58.2  0.344828

QC Failures (>1% difference): 2 out of 10


In [21]:
# PII check: any phone numbers or emails in the notes?
for i, note in enumerate(manual['Inspection_Notes']):
    has_phone = bool(pd.Series([note]).str.contains(r'\b\d{10}\b', regex=True).iloc[0])
    has_email = bool(pd.Series([note]).str.contains(r'\S+@\S+', regex=True).iloc[0])
    
    if has_phone or has_email:
        print(f'PII found in row {i}: "{note[:80]}..."')

total_pii = manual['Inspection_Notes'].str.contains(r'\b\d{10}\b|\S+@\S+', regex=True).sum()
print(f'\nTotal notes with PII: {total_pii}')

PII found in row 0: "Inspection passed. Signed off by Vikram (vikram.s@gov.in, 9090909090)...."
PII found in row 9: "System broken. Call Amit at 9876543210 or amit@email.com...."

Total notes with PII: 2


---
## Summary
All the issues we found that need cleaning.

In [22]:
print('ISSUES FOUND:')
print(f'  1. Timestamps with hour=24:        {has_24.sum()}')
print(f'  2. Timestamps with UTC:             {has_utc.sum()}')
print(f'  3. Timestamps with slash format:    {has_slash.sum()}')
print(f'  4. Timestamps in ISO format:        {has_iso.sum()}')
print(f'  5. Non-standard unit values:        {(raw["Unit"] != "ug/m3").sum()}')
print(f'  6. Non-standard status values:      {(~raw["Status"].isin(["OK","FAULT","MAINT"])).sum()}')
print(f'  7. Bad lat/lon coordinates:         {(has_semicolon | has_negative999).sum()}')
print(f'  8. Missing Record_ID gaps:          {len(missing)}')
print(f'  9. Manual entry QC failures:        {qc_fail.sum()}')
print(f' 10. Notes with personal info (PII):  {total_pii}')
print(f'\nNext step: 02_cleaning.ipynb')

ISSUES FOUND:
  1. Timestamps with hour=24:        155
  2. Timestamps with UTC:             283
  3. Timestamps with slash format:    269
  4. Timestamps in ISO format:        96
  5. Non-standard unit values:        1129
  6. Non-standard status values:      1712
  7. Bad lat/lon coordinates:         58
  8. Missing Record_ID gaps:          350
  9. Manual entry QC failures:        2
 10. Notes with personal info (PII):  2

Next step: 02_cleaning.ipynb
